## Rebuild filtered_complaints.csv

In [4]:
import pandas as pd

## Choose 5 Product Categories

In [5]:
selected_products = [
    "Credit card",
    "Mortgage",
    "Debt collection",
    "Checking or savings account",
    "Vehicle loan or lease"
]

selected_products


['Credit card',
 'Mortgage',
 'Debt collection',
 'Checking or savings account',
 'Vehicle loan or lease']

## Create Empty List

In [6]:
chunks = []

## Filter the Huge File in Chunks

In [7]:
for chunk in pd.read_csv(
    "../data/raw/complaints.csv",
    chunksize=100000,
    low_memory=False
):
    filtered_chunk = chunk[
        chunk["Product"].isin(selected_products)
    ]

    chunks.append(filtered_chunk)

print("Finished filtering")

Finished filtering


## Combine Results

In [8]:
df = pd.concat(chunks, ignore_index=True)

print(df.shape)

(1812272, 18)


## Verify Categories

In [9]:
print(df["Product"].value_counts())

Product
Debt collection                799197
Mortgage                       422254
Checking or savings account    291178
Credit card                    226686
Vehicle loan or lease           72957
Name: count, dtype: int64


## Remove Empty Narratives

In [10]:
df = df.dropna(
    subset=["Consumer complaint narrative"]
)

print(df.shape)

(726799, 18)


## Create clean_text

In [11]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [12]:
df["clean_text"] = df[
    "Consumer complaint narrative"
].apply(clean_text)

## Save the New Dataset

In [13]:
df.to_csv(
    "../data/filtered_complaints.csv",
    index=False
)

print("Saved successfully")

Saved successfully


## Verify the Saved File

In [14]:
df_check = pd.read_csv(
    "../data/filtered_complaints.csv"
)

print(df_check["Product"].value_counts())

Product
Debt collection                336076
Checking or savings account    140319
Mortgage                       130160
Credit card                     80667
Vehicle loan or lease           39577
Name: count, dtype: int64


## Create Stratified Sample

In [22]:
sample_df = df.groupby(
    "Product",
    group_keys=False
).sample(
    frac=0.02,
    random_state=42
)

## Check Sample Size

In [23]:
print(len(sample_df))

14536


In [24]:
print(sample_df["Product"].value_counts())

Product
Debt collection                6722
Checking or savings account    2806
Mortgage                       2603
Credit card                    1613
Vehicle loan or lease           792
Name: count, dtype: int64
